In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv("diamonds.csv")

In [3]:
df.head()

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [4]:
df = df.drop(["Unnamed: 0"], axis=1)

In [5]:
df = df.drop(df[df["x"]==0].index)
df = df.drop(df[df["y"]==0].index)
df = df.drop(df[df["z"]==0].index)

In [6]:
df = df[(df["depth"]<75)&(df["depth"]>45)]
df = df[(df["table"]<80)&(df["table"]>40)]
df = df[(df["y"]<30)]
df = df[(df["z"]<30)&(df["z"]>2)]

In [7]:
X= df.drop(["price"],axis =1)
y= df["price"]

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.25, random_state=15)

In [9]:
from sklearn.preprocessing import LabelEncoder

In [10]:
encoders={}
for col in ['cut','color','clarity']:
    encoders[col]=LabelEncoder()
    X_train[col]=encoders[col].fit_transform(X_train[col])
    X_test[col]=encoders[col].transform(X_test[col])

In [11]:
from sklearn.preprocessing import StandardScaler

In [12]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [13]:
from sklearn.svm import SVR

In [14]:
svr=SVR(C=1000,gamma=0.1,kernel='rbf')

In [15]:
svr.fit(X_train_scaled,y_train)
y_pred=svr.predict(X_test_scaled)

In [16]:
from sklearn.metrics import r2_score

In [17]:
score=r2_score(y_test,y_pred)
print("R2 Score: ",score)

R2 Score:  0.9452198431184519


In [18]:
import pickle

In [19]:
with open("diamond_model_complete.pkl","wb") as f:
    pickle.dump(
        {
            'model' : svr,
            'encoders' : encoders,
            'scaler' : scaler
        },f)

In [20]:
pd.DataFrame(X_test_scaled).to_csv("testdatascaled.csv",index=False)